# Stage1 collapse diagnostic
Four fixed-input FP32 controls. No deployment weights.

In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, tempfile, time, zipfile
STARTED = time.monotonic()
WORK = Path(tempfile.mkdtemp(prefix='stage1-training-', dir='/kaggle/working'))
items = list(Path('/kaggle/input').rglob('stage1-training-assets.json'))
if len(items) != 1: raise ValueError('Expected exactly one Stage1 training asset manifest')
asset = items[0].parent
for name, expected in json.loads(items[0].read_text()).items():
    with (asset / name).open('rb') as handle:
        actual = hashlib.file_digest(handle, 'sha256').hexdigest()
    if actual != expected: raise ValueError('Bundle checksum mismatch')
with zipfile.ZipFile(asset / 'stage1-training.bin') as archive:
    if any(not (WORK / name).resolve().is_relative_to(WORK.resolve()) for name in archive.namelist()): raise ValueError('ZIP path escape')
    archive.extractall(WORK)
config = json.loads((WORK / 'run_config.json').read_text())
import shutil
candidates=list(Path('/kaggle/input').rglob('diagnostic-code.json'))
if len(candidates)!=1:raise ValueError('Expected one diagnostic code manifest')
diag=candidates[0].parent/'stage1-optimization.py'
if hashlib.sha256(diag.read_bytes()).hexdigest()!=json.loads(candidates[0].read_text())['sha256']:raise ValueError('Diagnostic code checksum mismatch')
shutil.copyfile(diag,WORK/'src/diagnose_stage1_optimization.py')
(WORK / 'input-assets.json').write_text(items[0].read_text())
VENV = Path(tempfile.mkdtemp(prefix='stage1-pinned-', dir='/tmp'))
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(VENV)], check=True)
PYTHON = str(VENV / 'bin/python')
failure = None
try:
    install_started = time.monotonic()
    with (WORK / 'install.log').open('w') as log:
        installed = subprocess.run([sys.executable, '-m', 'pip', '--python', PYTHON, 'install', '--no-cache-dir', '-r', str(WORK / 'requirements.txt')], stdout=log, stderr=subprocess.STDOUT, timeout=600)
    (WORK / 'install.json').write_text(json.dumps(dict(exit_code=installed.returncode, seconds=time.monotonic()-install_started), indent=2))
    if installed.returncode: raise RuntimeError('Pinned package installation failed')
    deadline = 3500 if config['mode'] == 'trial' else 35400
    remaining = int(deadline - (time.monotonic() - STARTED))
    budget = min(config['max_seconds'], remaining - 240)
    if budget < 300: raise RuntimeError('Insufficient training budget')
    command = [PYTHON, '-u', 'src/diagnose_stage1_optimization.py', '--dataset-dir', 'dataset', '--output-dir', 'training', '--max-seconds', str(min(2400,budget))]
    with (WORK / 'training.log').open('w') as log:
        process = subprocess.Popen(command, cwd=WORK, stdout=log, stderr=subprocess.STDOUT)
        while process.poll() is None:
            if time.monotonic() - STARTED > deadline:
                process.kill(); process.wait(); raise TimeoutError('GPU run budget exceeded')
            time.sleep(30)
            print((WORK / 'training.log').read_text()[-1600:], flush=True)
        if process.returncode: raise RuntimeError('Stage1 training failed')
except Exception as error:
    failure = repr(error)
finally:
    (WORK / 'runner.json').write_text(json.dumps(dict(failure=failure, seconds=time.monotonic()-STARTED, config=config), indent=2))
    with zipfile.ZipFile('/kaggle/working/stage1-optimization-result.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
        for name in ['training', 'src']:
            for path in sorted((WORK / name).glob('*')):
                if path.is_file(): archive.write(path, path.relative_to(WORK))
        for name in ['training.log', 'install.log', 'install.json', 'runner.json', 'run_config.json', 'input-assets.json', 'dataset/generation_report.json', 'dataset/generation_manifest.csv', 'dataset/phone_holdout.csv', 'dataset/excluded_sources.csv']:
            if (WORK / name).exists(): archive.write(WORK / name, name)
print('Runner seconds', time.monotonic()-STARTED, 'failure', failure)
if failure: raise RuntimeError(failure)
